In [ ]:
# TODO: Implement this cell


## Cell 1: Setup MLflow Tracking Server

Start a local MLflow tracking server to log experiments.

**Why MLflow?**
- Centralized experiment history
- Automatic metric visualization
- Model versioning and staging
- Reproducibility guarantees

**Setup:**
```bash
# Terminal 1: Start tracking server
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns
```

We'll connect to this server from our training code.

In [ ]:
# TODO: Implement this cell


## Cell 2: Log First Experiment (Baseline)

Run baseline BERT fine-tuning and log everything to MLflow:
- **Hyperparameters**: learning_rate, batch_size, epochs, warmup_steps
- **Metrics**: accuracy, f1_score, loss (logged per epoch)
- **Artifacts**: model checkpoint, confusion matrix plot

**Tracking best practices:**
- Log system info (GPU, CUDA version)
- Tag runs with metadata (engineer name, branch)
- Save reproducible environment snapshot

In [ ]:
# TODO: Implement this cell


## Cell 3: Hyperparameter Sweep (12 Experiments)

Run grid search over hyperparameter space:
- Learning rates: `[3e-5, 5e-5, 7e-5]`
- Batch sizes: `[16, 32]`
- Warmup steps: `[250, 500]`

**Total: 3 × 2 × 2 = 12 runs**

MLflow automatically tracks all runs, enabling:
- Parallel coordinates plots
- Metric comparison tables
- Best model selection by objective

**Real-world note:** This would run on multiple GPUs in parallel. Here we simulate serially for demonstration.

In [ ]:
# TODO: Implement this cell


## Cell 4: MLflow UI Walkthrough

**Navigate to MLflow UI:** `http://127.0.0.1:5000`

**Key features demonstrated:**

1. **Run comparison table**
   - Sort by accuracy/F1 descending
   - Filter by tags (e.g., `engineer = "alice"`)
   - Select multiple runs → Compare

2. **Parallel coordinates plot**
   - Visualize hyperparameter impact
   - Drag axes to reorder
   - Brush to filter runs by metric range
   - Identify: high accuracy = high LR + small batch size

3. **Metric plots**
   - Plot accuracy vs. epoch for all runs
   - Overlay curves to compare convergence
   - Identify early stopping opportunities

4. **Artifact browser**
   - View confusion matrices side-by-side
   - Download model checkpoints
   - Inspect logged code versions

**Best run detection (visual):** Look for darkest line in parallel coords plot with high test_accuracy.

In [ ]:
# TODO: Implement this cell


## Cell 5: Search API (Find Best Run)

Use MLflow Search API to programmatically find best runs by metric.

**Query syntax:**
- `metrics.test_accuracy > 0.85`
- `params.learning_rate = '5e-5'`
- `tags.engineer = 'alice'`

**Use cases:**
- Automatic model selection for deployment
- Hyperparameter analysis (which params correlate with high accuracy?)
- Team dashboards (who produced best models this sprint?)

In [ ]:
# TODO: Implement this cell


## Cell 6: Model Registry (Register Best Model)

Register the best model in MLflow Model Registry:
- **Purpose**: Centralized model lifecycle management
- **Stages**: None → Staging → Production → Archived
- **Benefits**: Version control, access control, audit trail

**Workflow:**
1. Register model from best run
2. Tag as "staging" for QA validation
3. Add description and metadata
4. Notify team for approval

**Model Registry vs. Experiment Tracking:**
- Experiments: All training runs (incl. failures)
- Registry: Production-worthy models only

In [ ]:
# TODO: Implement this cell


## Cell 7: Load Model from Registry (Inference)

Load model from registry and run inference on test set.

**Registry benefits:**
- Load by stage: `models:/{model_name}/Staging`
- Load by version: `models:/{model_name}/{version}`
- Automatic lineage tracking (which run produced this model?)
- Deployment-ready artifact URI

**Production pattern:**
```python
# Serving code always loads from "Production" stage
model_uri = f"models:/{model_name}/Production"
model = mlflow.pytorch.load_model(model_uri)
```

In [ ]:
# TODO: Implement this cell


## Cell 8: Promote Model to Production

After QA validation passes, promote model from Staging → Production.

**Promotion workflow:**
1. QA team validates model on staging environment
2. Product owner approves metrics (accuracy, latency, fairness)
3. ML engineer promotes to Production stage
4. Archive previous production version

**Governance:**
- Audit trail: Who promoted when?
- Rollback capability: Revert to previous version
- Notifications: Alert team on production updates

In [ ]:
# TODO: Implement this cell


## Cell 9: DVC Setup (Version Training Dataset)

Use **DVC (Data Version Control)** to version large training datasets alongside code.

**Why DVC?**
- Git doesn't handle large files well (datasets, models)
- DVC stores data in remote storage (S3, GCS, Azure Blob)
- Lightweight Git commits + data snapshots = reproducibility

**Setup:**
```bash
# Initialize DVC
dvc init

# Add remote storage (e.g., S3 bucket)
dvc remote add -d myremote s3://ml-datasets-bucket/dvc-store

# Track dataset
dvc add data/imdb-25k.csv

# Commit DVC metadata (not data itself)
git add data/imdb-25k.csv.dvc data/.gitignore
git commit -m "Track IMDb dataset with DVC"

# Push data to remote
dvc push
```

**Result:** Dataset is versioned like code. Checkout old commit → `dvc pull` → get exact data.

In [ ]:
# TODO: Implement this cell


## Cell 10: Reproduce Experiment from Run ID

Full experiment reproduction workflow:
1. Fetch run metadata from MLflow
2. Retrieve code version (Git commit)
3. Pull dataset version (DVC)
4. Rerun training with exact hyperparameters

**Reproducibility guarantees:**
- **Code**: Git commit hash
- **Data**: DVC version hash
- **Hyperparameters**: MLflow logged params
- **Environment**: Conda env or Docker image

**Use cases:**
- Regulatory audits (FDA, financial services)
- Debugging model regressions
- Transfer learning from old experiments
- Paper submissions (reviewers can reproduce results)

In [ ]:
# TODO: Implement this cell
